# Chapter 1 — Notebook 1: Streaming ATLAS Open Data

**Goals**

- Stream the real ATLAS Open Data `tt̄` sample with `atlasopenmagic` + `uproot`.
- List the available branches and recognise the leptons / jets / MET / truth structure.
- Make your first histogram (lepton pT). All momenta/energies are in **GeV**.

**Physics context.** Each event is a simulated (or real) proton–proton collision. 
We use the `3J1LMET30` skim (≥3 jets + 1 tight lepton + MET>30) which enriches the 
semileptonic `tt̄` topology. Read [`data/README.md`](../data/README.md) for the branch dictionary.

In [ ]:
%matplotlib inline
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt

from topmass import io, kinematics, selection, plotting, fitting, neutrino, pairing, weights, style
from topmass.constants import M_W, M_TOP

> 🧭 **How these notebooks work.** Cells marked **✏️ Your turn** already run as they are — you do
> **not** start from a blank cell. Change only the line(s) flagged with `# ✏️`, then re-run the cell
> with **Shift+Enter** and look at how the output changes. Each task ends with a short physics
> question to answer in your report, plus an optional **Stretch** for one extra tweak.

## Worked example: stream the signal sample

The first call streams over the network and caches the result under `.cache/`; 
later calls read the cache. Keep `fraction` small while exploring.

In [ ]:
io.setup()                                  # select release 2025e-13tev-beta
samples = io.build_samples()                # skim '3J1LMET30', https
events = io.load_process('ttbar', samples, fraction=0.1)
print('Number of events:', len(events))

In [ ]:
print('Branches:', events.fields[:10], '...')
print('Leptons in first 5 events:', events.lep_n[:5].tolist())
print('Leading-lepton pT [GeV]:', events.lep_pt[:5, 0].tolist())

## Worked example: leading-lepton-pT histogram

In [ ]:
lead_lep_pt = events.lep_pt[:, 0]   # leptons are a jagged collection
fig, ax = plt.subplots()
ax.hist(ak.to_numpy(lead_lep_pt), bins=500, range=(0, 200))
ax.set_xlabel(r'leading $p_T^\ell$  [GeV]')
ax.set_ylabel('Events')

## ✏️ Your turn 1.1

▶️ This cell already runs. Change the line(s) marked `# ✏️` and re-run (**Shift+Enter**).

It plots the missing transverse energy (`events.met`) and the jet multiplicity (`events.jet_n`)
side-by-side. At what MET value does the distribution peak? Why is it non-zero on average for `tt̄`?

> **Stretch (optional):** set `N_BINS = 100` for finer structure, or widen `MET_RANGE` to see the tail.

In [ ]:
N_BINS    = 50          # ✏️ try 25, 50, 100
MET_RANGE = (0, 200)    # ✏️ try (0, 300) to see the tail

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(ak.to_numpy(events.met), bins=N_BINS, range=MET_RANGE)
axes[0].set_xlabel('MET [GeV]'); axes[0].set_ylabel('Events')
axes[1].hist(ak.to_numpy(events.jet_n), bins=range(0, 12))
axes[1].set_xlabel('number of jets'); axes[1].set_ylabel('Events')

## ✏️ Your turn 1.2

▶️ Change the marked line and re-run.

This streams a background process and overlays its leading-lepton-pT **shape** on the signal
(both normalised to unit area, so only the shapes are compared). Do signal and background peak at
different lepton momenta?

> **Stretch (optional):** set `VARIABLE = 'met'` to compare the MET shapes instead.

In [ ]:
OTHER    = 'single_top'   # ✏️ try 'single_top', 'diboson', 'data'
VARIABLE = 'lep_pt'       # ✏️ stretch: try 'met' (a per-event scalar)

other = io.load_process(OTHER, samples, fraction=0.1)

def values(ev):
    v = ev[VARIABLE]
    return ak.to_numpy(v[:, 0] if v.ndim > 1 else v)

plt.hist(values(events), bins=50, range=(0, 200), density=True, histtype='step', label='ttbar')
plt.hist(values(other),  bins=50, range=(0, 200), density=True, histtype='step', label=OTHER)
plt.xlabel(VARIABLE + '  (leading)'); plt.ylabel('normalised'); plt.legend()

## Wrap-up

1. Which branches are reconstruction-level and which are truth-level (`truth_*`)?
2. Why are `lep_pt` and `jet_pt` *lists* per event rather than single numbers?